## 7. Power BI export


Preparing the tables used by the Power BI dashboards.

The demand table includes actual demand, predictions from the model
selected in 5_modelling and the yesterday-demand baseline.
The currently selected model is LightGBM.

The exports contain:

- **Demand:** actual demand, model and baseline predictions, errors
- **Calendar and time:** dates, seasons, day types, half-hourly periods
- **Anomaly context:** all evaluated half-hours and their predictions
- **Anomaly flags:** one row per flagged half-hour per detection method
- **Anomaly groups:** summaries of consecutive robust z-score flags
- **Model scorecard:** evaluation metrics and recorded runtimes
- **Demand drivers:** weather, embedded generation, calendar information

Date and settlement-period keys connect the relevant tables in Power BI.

### 1. Setup

In [1]:
import sqlite3   
from pathlib import Path   
import numpy as np   
import pandas as pd   

CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DB_PATH  = PROJECT_ROOT / "database" / "energy.db"
OUT_DIR  = PROJECT_ROOT / "powerbi" / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

conn=sqlite3.connect(DB_PATH)
print(f"Database: {DB_PATH}")
print(f"Export to: {OUT_DIR}")

#loading model selection from 5_modelling
scorecard_path = (
    PROJECT_ROOT / "data" / "processed" / "model_comparison.csv"
)
scorecard = pd.read_csv(scorecard_path)

selected_mask = (
    scorecard["selected_on_validation"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "1.0"])
)

selected_models = scorecard.loc[selected_mask, "model"]

if len(selected_models) != 1:
    raise ValueError(
        "Expected exactly one model selected on validation. "
        "Check the model_comparison.csv export from Notebook 5."
    )

WINNER = selected_models.iloc[0]
BASELINE = "Baseline: persistence (yesterday)"

print(f"Selected model: {WINNER}")
print(f"Comparison baseline: {BASELINE}")

def export(dframe, name):
    path=OUT_DIR / f"{name}.csv"
    dframe.to_csv(path, index=False)
    print(f"  {name+'.csv':<28} {len(dframe):>8,} rows  {path.stat().st_size/1_000_000:>6.2f} MB")
    return dframe


Database: c:\Users\marco\Documents\PROJECTS\New folder\database\energy.db
Export to: c:\Users\marco\Documents\PROJECTS\New folder\powerbi\data
Selected model: LightGBM
Comparison baseline: Baseline: persistence (yesterday)


### 2. Demand fact table

One row per half-hour, carrying the actual demand, the chosen model's prediction, the baseline's prediction, and the residual. A single `date_key` (the ISO date) and `settlement_period` let Power BI join this to the calendar and time dimensions.

In [2]:
keys = ["date_key", "settlement_period"]

#Loading actual demand for all available dates
actuals = pd.read_sql_query("""
    SELECT settlement_date AS date_key,
           settlement_period,
           tsd_mw AS actual_mw
    FROM demand_readings
    WHERE settlement_period BETWEEN 1 AND 48
    ORDER BY settlement_date, settlement_period;
""", conn)

#Loading test-period predictions from selected model and yesterday's baseline
preds = pd.read_sql_query("""
    SELECT settlement_date AS date_key,
           settlement_period,
           model_name,
           predicted_demand_mw
    FROM model_predictions
    WHERE data_split = 'test'
      AND model_name IN (?, ?)
    ORDER BY settlement_date, settlement_period;
""", conn, params=[WINNER, BASELINE])

#Using matching date and period formats when joining tables
for frame in (actuals, preds):
    frame["date_key"] = (
        pd.to_datetime(frame["date_key"]).dt.strftime("%Y-%m-%d")
    )
    frame["settlement_period"] = (
        pd.to_numeric(frame["settlement_period"], errors="raise")
        .astype(int)
    )

assert not actuals.duplicated(keys).any(), "Duplicate actual observations."
assert not preds.duplicated(keys + ["model_name"]).any(), (
    "Duplicate model predictions."
)
assert set(preds["model_name"]) == {WINNER, BASELINE}, (
    "Missing selected-model or baseline predictions."
)

#Putting model and baseline predictions into separate columns
preds_wide = (
    preds.pivot(
        index=keys,
        columns="model_name",
        values="predicted_demand_mw"
    )
    .rename(columns={
        WINNER: "predicted_mw",
        BASELINE: "baseline_mw"
    })
    .reset_index()
)
preds_wide.columns.name = None

assert preds_wide[["predicted_mw", "baseline_mw"]].notna().all().all(), (
    "Model and baseline predictions do not cover the same observations."
)

fct = actuals.merge(
    preds_wide,
    on=keys,
    how="left",
    validate="one_to_one"
)

assert fct["predicted_mw"].notna().sum() == len(preds_wide), (
    "Some predictions have no matching actual-demand observation."
)

#Calculating errors as actual demand - predicted demand
fct["residual_mw"] = fct["actual_mw"] - fct["predicted_mw"]
fct["abs_residual_mw"] = fct["residual_mw"].abs()
fct["baseline_resid_mw"] = fct["actual_mw"] - fct["baseline_mw"]

#Converting settlement periods into half-hourly timestamps
fct["datetime"] = (
    pd.to_datetime(fct["date_key"])
    + pd.to_timedelta(
        (fct["settlement_period"] - 1) * 30,
        unit="m"
    )
)

export(fct, "fct_demand")

evaluated = fct.loc[fct["predicted_mw"].notna()]

print(f"\nSelected model: {WINNER}")
print(f"Actual-demand rows: {len(fct):,}")
print(f"Test prediction rows: {len(evaluated):,}")
print(
    f"Test period: {evaluated['date_key'].min()} "
    f"to {evaluated['date_key'].max()}"
)
print(f"Model MAE: {evaluated['abs_residual_mw'].mean():,.2f} MW")
print(
    f"Baseline MAE: "
    f"{evaluated['baseline_resid_mw'].abs().mean():,.2f} MW"
)

  fct_demand.csv                 52,602 rows    3.67 MB

Selected model: LightGBM
Actual-demand rows: 52,602
Test prediction rows: 17,510
Test period: 2025-01-01 to 2025-12-31
Model MAE: 1,057.09 MW
Baseline MAE: 2,008.53 MW


In [3]:
#Loading the historical predictions generated in 6_adv_modelling
oof_preds = pd.read_sql_query("""
    SELECT settlement_date AS date_key,
           settlement_period,
           model_name,
           predicted_demand_mw AS expected_demand_mw
    FROM model_predictions
    WHERE data_split = 'validation'
      AND model_name LIKE '% (out-of-fold)'
    ORDER BY settlement_date, settlement_period;
""", conn)

if oof_preds.empty:
    raise ValueError("No out-of-fold predictions found. Run Notebook 6 first.")

if oof_preds["model_name"].nunique() != 1:
    raise ValueError("Expected one out-of-fold model in the database.")

oof_preds["date_key"] = (
    pd.to_datetime(oof_preds["date_key"]).dt.strftime("%Y-%m-%d")
)
oof_preds["settlement_period"] = (
    pd.to_numeric(oof_preds["settlement_period"], errors="raise")
    .astype(int)
)

assert not oof_preds.duplicated(keys).any(), (
    "Duplicate out-of-fold predictions."
)

#Matching predictions with actual demand for every evaluated half-hour
fct_anomaly_context = oof_preds.merge(
    actuals.rename(columns={"actual_mw": "actual_demand_mw"}),
    on=keys,
    how="left",
    validate="one_to_one"
)

assert fct_anomaly_context[
    ["actual_demand_mw", "expected_demand_mw"]
].notna().all().all(), "Missing actual or expected demand."

fct_anomaly_context["residual_mw"] = (
    fct_anomaly_context["actual_demand_mw"]
    - fct_anomaly_context["expected_demand_mw"]
)

#Creating half-hourly timestamps without adjusting for cloc changes
fct_anomaly_context["datetime"] = (
    pd.to_datetime(fct_anomaly_context["date_key"])
    + pd.to_timedelta(
        (fct_anomaly_context["settlement_period"] - 1) * 30,
        unit="m"
    )
)

export(fct_anomaly_context, "fct_anomaly_context")

print(
    f"\nOut-of-fold model: "
    f"{fct_anomaly_context['model_name'].iloc[0]}"
)
print(f"Evaluated half-hours: {len(fct_anomaly_context):,}")
print(
    f"Coverage: {fct_anomaly_context['date_key'].min()} "
    f"to {fct_anomaly_context['date_key'].max()}"
)
print(
    f"Out-of-fold MAE: "
    f"{fct_anomaly_context['residual_mw'].abs().mean():,.1f} MW"
)

  fct_anomaly_context.csv        42,700 rows    4.62 MB

Out-of-fold model: LGBMRegressor (out-of-fold)
Evaluated half-hours: 42,700
Coverage: 2023-07-26 to 2025-12-31
Out-of-fold MAE: 1,104.1 MW


### 3. Calendar dimension

One row per date, carrying every attribute a slicer might use: year, month, season, day type, holiday flags.

date_key is the join column back to the fact table. 'date' is included too (as a real date type)


In [4]:
dim_calendar=pd.read_sql_query("""
    SELECT calendar_date AS date_key,
            calendar_date AS date,
            year, month, month_name, day_of_month, day_of_year,
            day_of_week, day_name, season, is_weekend,
            is_bank_holiday, bank_holiday_name, is_school_holiday,
            is_working_day, is_clock_change_day
    FROM calendar_info
    ORDER BY calendar_date;
""", conn)

#Day-type labels for slicers
def day_type(row):
    if row["is_bank_holiday"]: return "Bank holiday"
    if row["is_weekend"]: return "Weekend"
    return "Working day"
dim_calendar["day_type"]=dim_calendar.apply(day_type, axis=1)

# Month-year sort key to sort chronologically
dim_calendar["month_year"] = pd.to_datetime(dim_calendar["date"]).dt.strftime("%b %Y")
dim_calendar["mont_year_sort"]=(dim_calendar["year"]*100 + dim_calendar["month"])

export(dim_calendar, "dim_calendar")
dim_calendar.head()



  dim_calendar.csv                1,103 rows    0.11 MB


,date_key,date,year,month,month_name,day_of_month,day_of_year,day_of_week,day_name,season,is_weekend,is_bank_holiday,bank_holiday_name,is_school_holiday,is_working_day,is_clock_change_day,day_type,month_year,mont_year_sort
0,2023-01-01,2023-01-01,2023,1,January,1,1,6,Sunday,Winter,1,0,None,0,0,0,Weekend,Jan 2023,202301
1,2023-01-02,2023-01-02,2023,1,January,2,2,0,Monday,Winter,0,1,New Year’s Day,0,0,0,Bank holiday,Jan 2023,202301
2,2023-01-03,2023-01-03,2023,1,January,3,3,1,Tuesday,Winter,0,0,None,0,1,0,Working day,Jan 2023,202301
3,2023-01-04,2023-01-04,2023,1,January,4,4,2,Wednesday,Winter,0,0,None,0,1,0,Working day,Jan 2023,202301
4,2023-01-05,2023-01-05,2023,1,January,5,5,3,Thursday,Winter,0,0,None,0,1,0,Working day,Jan 2023,202301


### 4. Time_of_day dimension

One row per settlement period. Lets the dashboard slice by time of day and label axes in clock time
rather than period numbers 1–48.

In [5]:
dim_time=pd.read_sql_query("""
    SELECT settlement_period,
            hour, minute, time_label, hour_decimal, period_of_day, is_peak
    FROM time_periods
    WHERE settlement_period <=48
    ORDER BY settlement_period;
""", conn)

export(dim_time, "dim_time")
dim_time.head()

  dim_time.csv                       48 rows    0.00 MB


,settlement_period,hour,minute,time_label,hour_decimal,period_of_day,is_peak
0,1,0,0,00:00-00:30,0.0,Night,0
1,2,0,30,00:30-01:00,0.5,Night,0
2,3,1,0,01:00-01:30,1.0,Night,0
3,4,1,30,01:30-02:00,1.5,Night,0
4,5,2,0,02:00-02:30,2.0,Night,0


### 5. Anomalies fact table


One row per flagged half-hour per detection method. An observation flagged by multiple methods appears once for each method.
The dashboard uses residuals.csv for the complete set of evaluated half-hours, including observations that were not flagged.

In [6]:
fct_anomalies = pd.read_sql_query("""
    SELECT settlement_date AS date_key,
           settlement_period,
           method,
           actual_demand_mw,
           expected_demand_mw,
           residual_mw,
           anomaly_score
    FROM anomalies
    WHERE is_anomaly = 1
    ORDER BY settlement_date, settlement_period, method;
""", conn)

fct_anomalies["date_key"] = (
    pd.to_datetime(fct_anomalies["date_key"])
    .dt.strftime("%Y-%m-%d")
)
fct_anomalies["settlement_period"] = (
    pd.to_numeric(fct_anomalies["settlement_period"], errors="raise")
    .astype(int)
)

assert not fct_anomalies.duplicated(keys + ["method"]).any(), (
    "Duplicate flags for the same observation and method."
)

#Checking that actual demand, predictions, residuals match the full evaluation table
check_columns = [
    "actual_demand_mw",
    "expected_demand_mw",
    "residual_mw"
]

matched = fct_anomalies.merge(
    fct_anomaly_context[keys + check_columns],
    on=keys,
    how="left",
    validate="many_to_one",
    suffixes=("", "_context"),
    indicator=True
)

assert matched["_merge"].eq("both").all(), (
    "Some flags have no matching evaluated observation."
)

for column in check_columns:
    assert np.allclose(
        matched[column],
        matched[f"{column}_context"],
        rtol=0,
        atol=0.01
    ), f"Flagged observations disagree with context: {column}"

#Creating half-hourly timestamps without adjusting for clock changes
fct_anomalies["datetime"] = (
    pd.to_datetime(fct_anomalies["date_key"])
    + pd.to_timedelta(
        (fct_anomalies["settlement_period"] - 1) * 30,
        unit="m"
    )
)

fct_anomalies["direction"] = np.select(
    [
        fct_anomalies["residual_mw"] > 0,
        fct_anomalies["residual_mw"] < 0
    ],
    ["Above expected", "Below expected"],
    default="Equal to expected"
)

export(fct_anomalies, "fct_anomalies")

print("\nFlags by method:")
print(fct_anomalies.groupby("method").size().to_string())

print(
    "\nDistinct flagged half-hours: "
    f"{len(fct_anomalies[keys].drop_duplicates()):,}"
)
print("All flags match the out-of-fold context.")

  fct_anomalies.csv                 404 rows    0.05 MB

Flags by method:
method
consensus            24
isolation_forest    214
robust_zscore       166

Distinct flagged half-hours: 356
All flags match the out-of-fold context.


### 6. Anomaly events reference table

1 row per incident rather than per half-hour. Driver of the "top anomalies" table



In [7]:
events_path = (
    PROJECT_ROOT / "data" / "processed" / "anomaly_events.csv"
)

if not events_path.exists():
    raise FileNotFoundError(
        "anomaly_events.csv is missing. Run Notebook 6 first."
    )

dim_events = pd.read_csv(events_path)

required_columns = [
    "start_date", "start_period", "end_period",
    "n_periods", "duration_hours",
    "peak_residual_mw", "peak_actual_mw", "peak_expected_mw",
    "direction", "is_bank_holiday", "month"
]

missing = sorted(set(required_columns) - set(dim_events.columns))
assert not missing, f"Missing event columns: {missing}"

dim_events["start_date"] = (
    pd.to_datetime(dim_events["start_date"]).dt.strftime("%Y-%m-%d")
)

#Checking that the groups contain all half-hours flagged by robust z-score method
robust_count = fct_anomalies["method"].eq("robust_zscore").sum()

assert dim_events["n_periods"].sum() == robust_count, (
    "Event file and robust flags disagree. Rerun Notebook 6."
)

assert np.allclose(
    dim_events["duration_hours"],
    dim_events["n_periods"] * 0.5
), "Run durations do not match their half-hour counts."

dim_events["abs_peak_residual_mw"] = (
    dim_events["peak_residual_mw"].abs()
)

dim_events = dim_events.sort_values(
    "abs_peak_residual_mw", ascending=False
).reset_index(drop=True)

export(dim_events, "dim_anomaly_events")

print(f"\nConsecutive robust-flag runs: {len(dim_events):,}")
print(f"Half-hours represented: {dim_events['n_periods'].sum():,}")

  dim_anomaly_events.csv             52 rows    0.01 MB

Consecutive robust-flag runs: 52
Half-hours represented: 166


### 7. Model scorecard

Phase 5 comparison table for a dedicated methodology page.

In [8]:
scorecard_path = (
    PROJECT_ROOT / "data" / "processed" / "model_comparison.csv"
)

if not scorecard_path.exists():
    raise FileNotFoundError(
        "model_comparison.csv is missing. Run Notebook 5 first."
    )

scorecard = pd.read_csv(scorecard_path)

required_columns = [
    "model", "MAE", "RMSE", "MAPE", "R2",
    "fit_seconds", "skill_vs_baseline_%",
    "selected_on_validation"
]

missing = sorted(set(required_columns) - set(scorecard.columns))
assert not missing, f"Missing scorecard columns: {missing}"

assert not scorecard["model"].duplicated().any(), (
    "Duplicate models in the scorecard."
)

#Concverting the saved model-selection labels into True/False values
scorecard["selected_on_validation"] = (
    scorecard["selected_on_validation"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "1.0"])
)

selected = scorecard.loc[scorecard["selected_on_validation"]]

assert len(selected) == 1, "Expected exactly one selected model."
assert selected["model"].iloc[0] == WINNER, (
    "Scorecard selection differs from the demand export."
)

#Checking if MAE matches the exported predictions
exported_mae = fct.loc[
    fct["predicted_mw"].notna(), "abs_residual_mw"
].mean()

assert np.isclose(
    selected["MAE"].iloc[0],
    exported_mae,
    rtol=0,
    atol=0.01
), "Scorecard MAE does not match the exported test predictions."

scorecard = scorecard.sort_values("MAE").reset_index(drop=True)

export(scorecard, "model_scorecard")

print(f"\nScorecard entries: {len(scorecard)}")
print(f"Selected on validation: {WINNER}")
print(
    selected[
        ["model", "MAE", "MAPE", "skill_vs_baseline_%", "fit_seconds"]
    ].round(3).to_string(index=False)
)

  model_scorecard.csv                 9 rows    0.00 MB

Scorecard entries: 9
Selected on validation: LightGBM
   model      MAE  MAPE  skill_vs_baseline_%  fit_seconds
LightGBM 1057.089 3.723                47.37        2.589


### 8. Demand drivers table

In [9]:

fct_drivers = pd.read_sql_query("""
    SELECT settlement_date AS date_key,
           settlement_period,
           tsd_mw AS actual_mw,
           temperature_c,
           heating_degrees,
           cooling_degrees,
           embedded_solar_mw,
           embedded_wind_mw,
           underlying_demand_mw,
           is_working_day,
           is_bank_holiday
    FROM model_features_tbl
    WHERE settlement_period BETWEEN 1 AND 48
    ORDER BY settlement_date, settlement_period;
""", conn)

fct_drivers["date_key"] = (
    pd.to_datetime(fct_drivers["date_key"]).dt.strftime("%Y-%m-%d")
)
fct_drivers["settlement_period"] = (
    pd.to_numeric(fct_drivers["settlement_period"], errors="raise")
    .astype(int)
)

assert not fct_drivers.duplicated(keys).any(), (
    "Duplicate observations in the demand-drivers table."
)

#Creating half-hourly imestamps without adjusting for clock changes
fct_drivers["datetime"] = (
    pd.to_datetime(fct_drivers["date_key"])
    + pd.to_timedelta(
        (fct_drivers["settlement_period"] - 1) * 30,
        unit="m"
    )
)

#Labelling periods 20-32 as daytime (from 9:30 to 16:00)
fct_drivers["is_daytime"] = (
    fct_drivers["settlement_period"].between(20, 32).astype(int)
)

export(fct_drivers, "fct_drivers")

print(
    f"\nDriver coverage: {fct_drivers['date_key'].min()} "
    f"to {fct_drivers['date_key'].max()}"
)
print(f"Unique half-hours: {len(fct_drivers):,}")

  fct_drivers.csv                51,258 rows    4.43 MB

Driver coverage: 2023-01-29 to 2025-12-31
Unique half-hours: 51,258


### 9. Checking exports

In [10]:
print("EXPORTED FILES")
print("-" * 40)
total = 0
for f in sorted(OUT_DIR.glob("*.csv")):
    size = f.stat().st_size
    total += size
    print(f"  {f.name:<28} {size/1_000_000:>6.2f} MB")
print("-" * 40)
print(f"  {'TOTAL':<28} {total/1_000_000:>6.2f} MB")

conn.close()
print("\nConnection closed")

EXPORTED FILES
----------------------------------------
  dim_anomaly_events.csv         0.01 MB
  dim_calendar.csv               0.11 MB
  dim_time.csv                   0.00 MB
  fct_anomalies.csv              0.05 MB
  fct_anomaly_context.csv        4.62 MB
  fct_demand.csv                 3.67 MB
  fct_drivers.csv                4.43 MB
  model_scorecard.csv            0.00 MB
----------------------------------------
  TOTAL                         12.88 MB

Connection closed


In [11]:
path = PROJECT_ROOT / "data" / "processed" / "residuals.csv"
check = pd.read_csv(path)

print(f"File: {path.resolve()}")
print(f"Rows: {len(check):,}")
print(
    "Unique date/period pairs:",
    len(check.drop_duplicates(["settlement_date", "settlement_period"]))
)

File: C:\Users\marco\Documents\PROJECTS\New folder\data\processed\residuals.csv
Rows: 42,700
Unique date/period pairs: 42700
